# ⚽ Mission 13: Player Tracking & AI Performance Analysis

## From Seeing Artin to Understanding His Movement

In Mission 12, the AI Soccer Coach learned to detect players in a video frame.

Mission 13 asks a new question:

> **Where did Artin move, and what does that movement tell us?**

We will build this pipeline:

**Video → YOLO Tracking → Coordinates → Movement Features → Tactical Analysis → AI Coach**

The system will calculate:
- Distance covered
- Speed
- Movement zones
- Movement trajectory
- Estimated ball possession when ball coordinates are available
- AI coaching recommendations

Some values are **measured** from tracking data. Others, such as possession, are **estimates** and require reliable ball detection.

# Phase 1 — Detection vs. Tracking 👁️🏃

Detection looks at one frame:

```text
Frame 1 → Person, Person, Person
```

Tracking connects detections over time:

```text
Frame 1 → Player 4
Frame 2 → Player 4
Frame 3 → Player 4
```

The tracking ID lets us follow the same detected object.

For this learning exercise, we manually choose which tracking ID represents Artin.

Example:

```python
artin_id = 4
```

YOLO does **not** know that Player ID 4 is Artin.

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import cv2
import pandas as pd
import matplotlib.pyplot as plt
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

video_path = "Video1.mp4"

cap = cv2.VideoCapture(video_path)

fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

print("FPS:", fps)
print("Video size:", width, "x", height)

cap.release()

## Challenge 1 🏆

Change `video_path` to your soccer video.

Then identify:
- FPS
- Width
- Height

# Phase 2 — Find Tracking IDs

We will inspect the first part of the video and collect the IDs produced by the tracker.

The important setting is:

```python
persist=True
```

It tells the tracker to remember tracking information between frames.

In [ ]:
cap = cv2.VideoCapture(video_path)

frame_number = 0
detected_ids = []

while True:
    ret, frame = cap.read()

    if not ret:
        break

    frame_number += 1

    results = model.track(
        frame,
        persist=True,
        tracker="bytetrack.yaml",
        verbose=False
    )

    result = results[0]

    if result.boxes.id is not None:
        ids = result.boxes.id.cpu().tolist()

        for player_id in ids:
            player_id = int(player_id)

            if player_id not in detected_ids:
                detected_ids.append(player_id)

    if frame_number >= 100:
        break

cap.release()

print("Tracking IDs found:", detected_ids)

## Challenge 2 🏆

Inspect the IDs and decide which one is Artin.

For example:

```python
artin_id = 4
```

Remember: this is a manual identification step.

# Phase 3 — Track Artin's Position 📍

For every frame where Artin is detected, we record:

- Frame
- Time
- Player ID
- Pixel X
- Pixel Y

We use the center of the bounding box as the player's position:

```text
Center X = (x1 + x2) / 2
Center Y = (y1 + y2) / 2
```

In [ ]:
artin_id = 4

artin_positions = []

cap = cv2.VideoCapture(video_path)
frame_number = 0

while True:
    ret, frame = cap.read()

    if not ret:
        break

    frame_number += 1

    results = model.track(
        frame,
        persist=True,
        tracker="bytetrack.yaml",
        verbose=False
    )

    result = results[0]

    if result.boxes.id is not None:

        boxes = result.boxes.xyxy.cpu().tolist()
        ids = result.boxes.id.cpu().tolist()

        for box, player_id in zip(boxes, ids):

            player_id = int(player_id)

            if player_id == artin_id:

                x1, y1, x2, y2 = box

                center_x = (x1 + x2) / 2
                center_y = (y1 + y2) / 2

                time_seconds = (frame_number - 1) / fps

                artin_positions.append({
                    "frame": frame_number,
                    "time": time_seconds,
                    "player_id": player_id,
                    "pixel_x": center_x,
                    "pixel_y": center_y
                })

cap.release()

artin_df = pd.DataFrame(artin_positions)

print("Tracked positions:", len(artin_df))
artin_df.head()

# Phase 4 — Convert Pixels to Pitch Coordinates ⚽

We use the same educational conversion from Mission 12:

- Pitch length = 120 yards
- Pitch width = 80 yards

This is an approximation. A professional system would use camera calibration or homography.

In [ ]:
def convert_coordinates(pixel_x, pixel_y, frame_width, frame_height):

    pitch_x = (pixel_x / frame_width) * 120
    pitch_y = (pixel_y / frame_height) * 80

    return pitch_x, pitch_y

In [ ]:
pitch_x_values = []
pitch_y_values = []

for i in range(len(artin_df)):

    x, y = convert_coordinates(
        artin_df.loc[i, "pixel_x"],
        artin_df.loc[i, "pixel_y"],
        width,
        height
    )

    pitch_x_values.append(x)
    pitch_y_values.append(y)

artin_df["pitch_x"] = pitch_x_values
artin_df["pitch_y"] = pitch_y_values

artin_df.head()

# Phase 5 — Distance Covered 📏

For consecutive positions:

$$\text{Distance} = \sqrt{(x_2 - x_1)^2 + (y_2 - y_1)^2}$$
Because our pitch coordinates are in yards, our result is an **estimated distance in yards**.

* **Example:** Frame 1: $(53, 28) \rightarrow$ Frame 2: $(56, 31) \implies \text{Distance} = 4.2 \text{ yards}$
* **Total Distance:** Sum of all frame-to-frame movements.



In [ ]:
import math

artin_df["distance_yards"] = 0.0

for i in range(1, len(artin_df)):

    x1 = artin_df.loc[i - 1, "pitch_x"]
    y1 = artin_df.loc[i - 1, "pitch_y"]

    x2 = artin_df.loc[i, "pitch_x"]
    y2 = artin_df.loc[i, "pitch_y"]

    distance = math.sqrt(
        (x2 - x1) ** 2 +
        (y2 - y1) ** 2
    )

    artin_df.loc[i, "distance_yards"] = distance

total_distance_yards = artin_df["distance_yards"].sum()

print("Estimated distance:",
      round(total_distance_yards, 2),
      "yards")

# Phase 6 — Speed 🏃

Speed is:

```text
Speed = Distance / Time
```

We calculate speed in **yards per second**. We do not call this km/h because our simple coordinate system is measured in yards.


* **Example:** $\text{Distance} = 4\text{ meters}, \text{Time} = 0.5\text{ seconds} \implies \text{Speed} = 8\text{ m/s}$
* Reports **Average Speed** and **Maximum Speed**.

In [ ]:
artin_df["time_difference"] = artin_df["time"].diff()

artin_df["speed_yards_per_second"] = 0.0

for i in range(1, len(artin_df)):

    distance = artin_df.loc[i, "distance_yards"]
    dt = artin_df.loc[i, "time_difference"]

    if dt > 0:
        artin_df.loc[i, "speed_yards_per_second"] = distance / dt

average_speed = artin_df["speed_yards_per_second"].mean()
maximum_speed = artin_df["speed_yards_per_second"].max()

print("Average speed:",
      round(average_speed, 2),
      "yards/second")

print("Maximum speed:",
      round(maximum_speed, 2),
      "yards/second")

# Phase 7 — Movement Zones 🗺️

We divide the pitch into three simple zones:

```text
0 -------- 40 -------- 80 -------- 120
 Defense     Midfield      Final Third
```

We will calculate where Artin spent most of his tracked time.

In [ ]:
def get_zone(pitch_x):

    if pitch_x < 40:
        return "Defense"
    elif pitch_x < 80:
        return "Midfield"
    else:
        return "Final Third"

artin_df["zone"] = artin_df["pitch_x"].apply(get_zone)

zone_percent = (
    artin_df["zone"]
    .value_counts(normalize=True)
    * 100
)

print(zone_percent.round(1))

# Phase 8 — Visualize Artin's Movement 📈

In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(
    artin_df["pitch_x"],
    artin_df["pitch_y"],
    marker="."
)

plt.xlabel("Pitch Length (yards)")
plt.ylabel("Pitch Width (yards)")
plt.title("Artin's Movement Trajectory")

plt.xlim(0, 120)
plt.ylim(0, 80)
plt.grid()

plt.show()

# Phase 9 — Professional Pitch Visualization ⚽

In [ ]:
from mplsoccer import Pitch

pitch = Pitch(
    pitch_type="statsbomb",
    pitch_color="#aabb97",
    line_color="white"
)

fig, ax = pitch.draw(figsize=(10, 7))

pitch.plot(
    artin_df["pitch_x"],
    artin_df["pitch_y"],
    ax=ax,
    linewidth=2
)

pitch.scatter(
    artin_df["pitch_x"],
    artin_df["pitch_y"],
    ax=ax,
    s=15
)

plt.title("Artin's Movement on the Soccer Pitch")
plt.show()

# Phase 10 — Save the Tracking Dataset 📊

The dataset now contains much more information than the Mission 12 single-frame table.

We save it as:

```text
artin_tracking.csv
```

In [ ]:
artin_df.to_csv(
    "artin_tracking.csv",
    index=False
)

print("✅ Saved artin_tracking.csv")

# Phase 11 — Create a Performance Summary

Let's turn the tracking table into a small report that a coach can understand.

In [ ]:
most_common_zone = artin_df["zone"].value_counts().idxmax()

performance_summary = {
    "Player": "Artin",
    "Tracking ID": artin_id,
    "Tracked Frames": len(artin_df),
    "Distance Covered (yards)": round(total_distance_yards, 2),
    "Average Speed (yards/sec)": round(average_speed, 2),
    "Maximum Speed (yards/sec)": round(maximum_speed, 2),
    "Most Occupied Zone": most_common_zone
}

for key, value in performance_summary.items():
    print(f"{key}: {value}")

# Phase 12 — Ball Possession ⚽

Player tracking alone cannot tell us who has the ball.

We also need the ball's position.

For example:

```text
Artin: (53, 28)
Ball:  (54, 29)
```

If the ball is close enough, we can **estimate** possession.

This is only an estimate. A reliable possession system requires reliable ball detection and the same coordinate system for both player and ball.

In [ ]:
# Example ball coordinates for learning

ball_data = [
    {"frame": 1, "ball_x": 53.8, "ball_y": 28.5},
    {"frame": 2, "ball_x": 54.5, "ball_y": 29.3},
    {"frame": 3, "ball_x": 64.0, "ball_y": 38.0}
]

ball_df = pd.DataFrame(ball_data)

ball_df

In [ ]:
possession_df = artin_df[
    ["frame", "pitch_x", "pitch_y"]
].merge(
    ball_df,
    on="frame",
    how="inner"
)

possession_df["ball_distance"] = 0.0

for i in range(len(possession_df)):

    player_x = possession_df.loc[i, "pitch_x"]
    player_y = possession_df.loc[i, "pitch_y"]

    ball_x = possession_df.loc[i, "ball_x"]
    ball_y = possession_df.loc[i, "ball_y"]

    distance = math.sqrt(
        (player_x - ball_x) ** 2 +
        (player_y - ball_y) ** 2
    )

    possession_df.loc[i, "ball_distance"] = distance

possession_threshold = 2.0

possession_df["estimated_possession"] = (
    possession_df["ball_distance"] < possession_threshold
)

possession_df

## Challenge 3 🏆

Change:

```python
possession_threshold = 2.0
```

to:

```python
possession_threshold = 3.0
```

Does the estimated possession change?

This demonstrates that a simple rule can affect an AI system's result.

# Phase 13 — AI Coach Decision 🤖

Now we connect tracking to GenAI.

The important idea is:

**Do not ask Gemini to invent the statistics.**

Python calculates the measurements.

Gemini interprets those measurements and produces coaching advice.

```text
Computer Vision
      ↓
Tracking
      ↓
Measurements
      ↓
Gemini
      ↓
Coaching Decision
```

In [ ]:
coach_data = {
    "player": "Artin",
    "position": "Right Winger",
    "tracking_id": artin_id,
    "tracked_frames": len(artin_df),
    "distance_yards": round(total_distance_yards, 2),
    "average_speed_yards_per_second": round(average_speed, 2),
    "maximum_speed_yards_per_second": round(maximum_speed, 2),
    "most_occupied_zone": most_common_zone
}

coach_data

In [ ]:
prompt = f'''
You are an AI soccer coach.

Analyze this tracking data for Artin.

Player: {coach_data["player"]}
Position: {coach_data["position"]}
Distance covered: {coach_data["distance_yards"]:.1f} yards
Average speed: {coach_data["average_speed_yards_per_second"]:.2f} yards/second
Maximum speed: {coach_data["maximum_speed_yards_per_second"]:.2f} yards/second
Most occupied zone: {coach_data["most_occupied_zone"]}

Provide:
1. Movement strengths
2. Possible tactical weaknesses
3. One specific coaching decision
4. One training recommendation

Do not invent statistics that are not provided.
Clearly distinguish observations from recommendations.
'''

print(prompt)

# Connect Gemini

If your Gemini environment is already configured, send the prompt to the model.

Keep the API key out of the notebook source.

In [ ]:
import google.generativeai as genai

# Use your existing secure API-key setup.
# Example:
# genai.configure(api_key="YOUR_API_KEY")

ai_model = genai.GenerativeModel("gemini-1.5-flash")

response = ai_model.generate_content(prompt)

print(response.text)

# 🏆 Final Boss Challenge — Artin FC AI Tactical Analyst

Build an application that accepts:

- Soccer video
- Artin's tracking ID
- Artin's position

and produces:

## Tracking
- Frame
- Time
- Player ID
- Pixel X
- Pixel Y
- Pitch X
- Pitch Y

## Performance
- Distance covered
- Average speed
- Maximum speed
- Most occupied zone

## Visualization
- Movement trajectory
- Soccer pitch visualization
- Heatmap

## Ball Analysis
If a ball detector is available:
- Ball location
- Player-ball distance
- Estimated possession

## AI Coach
Gemini should provide:
- Strengths
- Tactical weaknesses
- A coaching decision
- A training recommendation

The final question your system should answer is:

> **"Where did Artin move, what does the tracking data show, and what should the coach recommend?"**

# Phase 14 — Build the AI Soccer Tracking App 🖥️⚽

Up to this point, you have built your player-tracking system inside a Google Colab.

Now we will turn that system into a **real application**.

The goal is to create:

## ⚽ Artin FC AI Player Tracking Center

The coach will be able to:

1. Enter the player's information
2. Upload a soccer video
3. Select a tracking ID
4. Track the player
5. Measure movement
6. Calculate distance and speed
7. Visualize the player's movement
8. Generate a heatmap
9. Identify the player's most occupied zone

The pipeline becomes:

**Video → YOLO Tracking → Player Coordinates → Movement Features → Dashboard → Tactical Visualization**



---

## Step 14.1 — Create the Streamlit Application

Instead of running the application inside this notebook, we will create a separate Python file.

Create a new file in the same project folder:

```text
tracking_app.py
```

Start the application with:

```python
import streamlit as st
import cv2
import pandas as pd
import tempfile
import math
import matplotlib.pyplot as plt

from ultralytics import YOLO
from mplsoccer import Pitch
```

Then configure the Streamlit page:

```python
st.set_page_config(
    page_title="Artin FC AI Tracking Center",
    page_icon="⚽",
    layout="wide"
)

st.title("⚽ Artin FC AI Tracking Center")

st.write(
    "Track a soccer player and analyze their movement."
)
```

Save the file.

Open your terminal and run:

```bash
streamlit run tracking_app.py
```

Your browser should open the application.

### 🏆 Challenge 1

Change the title and description to create your own version of the Artin FC application.

For example, you could create a different team name, application name, or description.

---



## Step 14.2 — Add Player Information

The coach needs to tell the application which player we want to analyze.

Add:

```python
player_name = st.text_input(
    "Player Name",
    value="Artin"
)

position = st.selectbox(
    "Player Position",
    [
        "Forward",
        "Midfielder",
        "Defender",
        "Winger"
    ]
)

artin_id = st.number_input(
    "Player Tracking ID",
    min_value=1,
    value=4,
    step=1
)
```

Now the application has three important pieces of information:

```text
Player Name
      ↓
Artin

Position
      ↓
Winger

Tracking ID
      ↓
4
```

### 🏆 Challenge 2

Change the default position and tracking ID to match the player in your video.

---



## Step 14.3 — Upload the Soccer Video

Now add a video uploader:

```python
uploaded_video = st.file_uploader(
    "Upload Soccer Video",
    type=["mp4", "mov", "avi"]
)
```

If a video has been uploaded, display it:

```python
if uploaded_video is not None:

    st.video(uploaded_video)

    st.success("Video uploaded successfully!")
```

At this point, the application should contain:

```text
⚽ Artin FC AI Tracking Center

Player Name
[ Artin ]

Player Position
[ Winger ▼ ]

Player Tracking ID
[ 4 ]

Upload Soccer Video
[ Choose a video ]

        Video Preview
```

---

## Step 14.4 — Load the Video

OpenCV needs a file path to read the uploaded video.

Inside:

```python
if uploaded_video is not None:
```

add:

```python
temp_file = tempfile.NamedTemporaryFile(
    delete=False,
    suffix=".mp4"
)

temp_file.write(
    uploaded_video.read()
)

temp_file.close()
```

Now OpenCV can open the temporary file:

```python
cap = cv2.VideoCapture(
    temp_file.name
)
```

Read the video information:

```python
fps = cap.get(
    cv2.CAP_PROP_FPS
)

width = int(
    cap.get(cv2.CAP_PROP_FRAME_WIDTH)
)

height = int(
    cap.get(cv2.CAP_PROP_FRAME_HEIGHT)
)

total_frames = int(
    cap.get(cv2.CAP_PROP_FRAME_COUNT)
)
```

Display the information:

```python
col1, col2, col3 = st.columns(3)

col1.metric(
    "Frames",
    total_frames
)

col2.metric(
    "FPS",
    round(fps, 1)
)

col3.metric(
    "Resolution",
    f"{width} × {height}"
)
```

---

## Step 14.5 — Load YOLO

Now load the YOLO model:

```python
model = YOLO("yolov8n.pt")
```

Add a tracking button:

```python
start_tracking = st.button(
    "▶️ Start Player Tracking"
)
```

The button is important.

Without the button, Streamlit may rerun the tracking code every time the application refreshes.

We want the coach to decide when tracking begins.

---



## Step 14.6 — Track the Selected Player

When the coach presses the button:

```python
if start_tracking:

    tracking_records = []

    frame_number = 0

    progress_bar = st.progress(0)
```

Now process the video:

```python
    while True:

        ret, frame = cap.read()

        if not ret:
            break

        frame_number += 1

        results = model.track(
            frame,
            persist=True,
            tracker="bytetrack.yaml",
            classes=[0],
            verbose=False
        )

        result = results[0]
```

Notice the addition of:

```python
classes=[0]
```

YOLO class `0` represents a **person**.

This means we are asking YOLO to track people rather than unrelated objects.

Now check whether tracking IDs exist:

```python
        if result.boxes.id is not None:

            boxes = result.boxes.xyxy.cpu().tolist()

            ids = result.boxes.id.cpu().tolist()
```

Loop through the detected players:

```python
            for box, player_id in zip(
                boxes,
                ids
            ):

                player_id = int(player_id)

                if player_id == artin_id:

                    x1, y1, x2, y2 = box

                    center_x = (
                        x1 + x2
                    ) / 2

                    center_y = (
                        y1 + y2
                    ) / 2
```

Save the tracking information:

```python
                    tracking_records.append({

                        "frame": frame_number,

                        "time": (
                            frame_number - 1
                        ) / fps,

                        "player_id": player_id,

                        "pixel_x": center_x,

                        "pixel_y": center_y
                    })
```

Update the progress bar:

```python
        progress_bar.progress(
            min(frame_number / total_frames, 1.0)
        )
```


## Step 14.7 — Create the Tracking DataFrame

After the video finishes:

```python
    cap.release()

    tracking_df = pd.DataFrame(
        tracking_records
    )
```

Check whether the player was actually found:

```python
    if tracking_df.empty:

        st.error(
            f"Tracking ID {artin_id} was not found."
        )

        st.stop()
```

Display the tracking data:

```python
    st.subheader(
        "📊 Player Tracking Data"
    )

    st.dataframe(
        tracking_df,
        use_container_width=True
    )
```

The coach should see data similar to:

```text
frame    time    player_id    pixel_x    pixel_y
1        0.00       4          850        430
2        0.04       4          855        432
3        0.08       4          861        437
```

This table is the foundation of the entire analysis system.

---

## Step 14.8 — Convert Pixel Coordinates

Now reuse the coordinate-conversion function from the earlier mission.

Add this function near the top of `tracking_app.py`:

```python
def convert_coordinates(
    pixel_x,
    pixel_y,
    frame_width,
    frame_height
):

    pitch_x = (
        pixel_x / frame_width
    ) * 120

    pitch_y = (
        pixel_y / frame_height
    ) * 80

    return pitch_x, pitch_y
```

Apply it to every tracked position:

```python
    pitch_x = []
    pitch_y = []

    for i in range(len(tracking_df)):

        x, y = convert_coordinates(
            tracking_df.loc[i, "pixel_x"],
            tracking_df.loc[i, "pixel_y"],
            width,
            height
        )

        pitch_x.append(x)
        pitch_y.append(y)

    tracking_df["pitch_x"] = pitch_x
    tracking_df["pitch_y"] = pitch_y
```

The application now transforms:

```text
Pixel Coordinates
        ↓
Pitch Coordinates
        ↓
Movement Analysis
```

---


## Step 14.9 — Calculate Distance 📏

Create a distance column:

```python
    tracking_df["distance_yards"] = 0.0
```

Calculate movement between consecutive tracked frames:

```python
    tracking_df["frame_gap"] = (
        tracking_df["frame"].diff()
    )

    for i in range(1, len(tracking_df)):

        if tracking_df.loc[i, "frame_gap"] != 1:
            continue

        x1 = tracking_df.loc[
            i - 1,
            "pitch_x"
        ]

        y1 = tracking_df.loc[
            i - 1,
            "pitch_y"
        ]

        x2 = tracking_df.loc[
            i,
            "pitch_x"
        ]

        y2 = tracking_df.loc[
            i,
            "pitch_y"
        ]

        distance = math.sqrt(
            (x2 - x1) ** 2 +
            (y2 - y1) ** 2
        )

        tracking_df.loc[
            i,
            "distance_yards"
        ] = distance
```

Calculate the total distance:

```python
    total_distance = (
        tracking_df[
            "distance_yards"
        ].sum()
    )
```

---


---



## Step 14.10 — Calculate Speed 🏃

Calculate the time difference:

```python
    tracking_df["time_difference"] = (
        tracking_df["time"].diff()
    )
```

Create the speed column:

```python
    tracking_df[
        "speed_yards_per_second"
    ] = 0.0
```

Calculate speed:

```python
    for i in range(1, len(tracking_df)):

        if tracking_df.loc[
            i,
            "frame_gap"
        ] != 1:

            continue

        distance = tracking_df.loc[
            i,
            "distance_yards"
        ]

        time_difference = tracking_df.loc[
            i,
            "time_difference"
        ]

        if (
            pd.notna(time_difference)
            and time_difference > 0
        ):

            speed = (
                distance /
                time_difference
            )

            tracking_df.loc[
                i,
                "speed_yards_per_second"
            ] = speed
```

---




## Step 14.11 — Display Player Statistics 📊

Calculate the statistics:

```python
    average_speed = (
        tracking_df[
            "speed_yards_per_second"
        ].mean()
    )

    maximum_speed = (
        tracking_df[
            "speed_yards_per_second"
        ].max()
    )
```

Now create the dashboard:

```python
    st.subheader(
        "📊 Player Performance"
    )

    col1, col2, col3 = st.columns(3)

    col1.metric(
        "Distance Covered",
        f"{total_distance:.1f} yards"
    )

    col2.metric(
        "Average Speed",
        f"{average_speed:.2f} yd/s"
    )

    col3.metric(
        "Maximum Speed",
        f"{maximum_speed:.2f} yd/s"
    )
```

The application has now moved from:

```text
"I can track a player."
```

to:

```text
"I can measure the player's movement."
```



---

## Step 14.12 — Add Movement Zones 🗺️

Reuse the zone function from Mission 13:

```python
    def get_zone(pitch_x):

        if pitch_x < 40:
            return "Defense"

        elif pitch_x < 80:
            return "Midfield"

        return "Final Third"
```

Apply it:

```python
    tracking_df["zone"] = (
        tracking_df["pitch_x"]
        .apply(get_zone)
    )
```

Find the most occupied zone:

```python
    most_common_zone = (
        tracking_df["zone"]
        .value_counts()
        .idxmax()
    )
```

Now add it to the dashboard:

```python
    col1, col2, col3, col4 = st.columns(4)

    col1.metric(
        "Distance Covered",
        f"{total_distance:.1f} yards"
    )

    col2.metric(
        "Average Speed",
        f"{average_speed:.2f} yd/s"
    )

    col3.metric(
        "Maximum Speed",
        f"{maximum_speed:.2f} yd/s"
    )

    col4.metric(
        "Most Occupied Zone",
        most_common_zone
    )
```

---

## Step 14.13 — Add the Movement Map 📈

Create a simple movement trajectory:

```python
    fig, ax = plt.subplots(
        figsize=(10, 6)
    )

    ax.plot(
        tracking_df["pitch_x"],
        tracking_df["pitch_y"]
    )

    ax.set_xlim(0, 120)
    ax.set_ylim(0, 80)

    ax.set_xlabel(
        "Pitch Length (yards)"
    )

    ax.set_ylabel(
        "Pitch Width (yards)"
    )

    ax.set_title(
        f"{player_name}'s Movement"
    )

    st.pyplot(fig)

    plt.close(fig)
```

The coach can now see the player's movement path.

---

## Step 14.14 — Add the Professional Pitch Visualization ⚽

Import `Pitch`:

```python
from mplsoccer import Pitch
```

Create the pitch:

```python
    pitch = Pitch(
        pitch_type="statsbomb",
        pitch_color="#aabb97",
        line_color="white"
    )
```

Draw it:

```python
    fig, ax = pitch.draw(
        figsize=(10, 7)
    )
```

Plot the player's trajectory:

```python
    pitch.plot(
        tracking_df["pitch_x"],
        tracking_df["pitch_y"],
        ax=ax,
        linewidth=2
    )
```

Add the tracking points:

```python
    pitch.scatter(
        tracking_df["pitch_x"],
        tracking_df["pitch_y"],
        ax=ax,
        s=15
    )

    ax.set_title(
        f"{player_name}'s Movement on the Soccer Pitch"
    )

    st.pyplot(fig)

    plt.close(fig)
```



---

## Step 14.15 — Add the Heatmap 🔥

Now we can reuse the heatmap concept from the previous mission.

Create the pitch:

```python
    pitch = Pitch(
        pitch_type="statsbomb",
        pitch_color="#aabb97",
        line_color="white"
    )

    fig, ax = pitch.draw(
        figsize=(10, 7)
    )
```

Calculate the occupied areas:

```python
    bin_statistic = pitch.bin_statistic(
        tracking_df["pitch_x"],
        tracking_df["pitch_y"],
        statistic="count",
        bins=(12, 8)
    )
```

Draw the heatmap:

```python
    pitch.heatmap(
        bin_statistic,
        ax=ax,
        cmap="Reds",
        alpha=0.6
    )
```

Add the player's movement points:

```python
    pitch.scatter(
        tracking_df["pitch_x"],
        tracking_df["pitch_y"],
        ax=ax,
        s=15
    )
```

Display the result:

```python
    ax.set_title(
        f"{player_name}'s Movement Heatmap"
    )

    st.pyplot(fig)

    plt.close(fig)
```

---



# 🏆 Final Boss Challenge — Artin FC AI Tactical Analyst

Build on the working tracking app.

The application should accept:
- Soccer video
- Player tracking ID
- Player position

and produce:
- Tracking table
- Distance covered
- Average and maximum speed
- Most occupied zone
- Movement trajectory
- Pitch visualization
- Heatmap

Ball analysis should only be added when a reliable ball detector is available.

The final question is:

> **Where did Artin move, what does the tracking data s

---

# ⭐ Super Challenge — Tactical Zone Analysis

Add another metric:

```text
Most Occupied Zone
```

The possible zones are:

```text
Defense
Midfield
Final Third
```

Use the player's pitch coordinates to determine the zone.

Then display:

```python
st.metric(
    "Most Occupied Zone",
    most_common_zone
)
```

### ⭐ Extra Challenge

Instead of only showing the most occupied zone, calculate the percentage of time the player spent in each zone.

For example:

```text
Defense:      12.4%
Midfield:     63.7%
Final Third:  23.9%
```

This turns the application from a simple tracking tool into a basic **tactical analysis system**.

---


# 🚀 Mission 13 Complete

You have moved from:

**Mission 12 — Vision 👁️**
> Where is the player?

to:

**Mission 13 — Tracking 🏃**
> Where did the player move?

and finally:

**AI Coaching 🤖**
> What should the coach recommend?

The next step is to make the analytics more sophisticated: heatmaps, tactical zones, attacking/defensive movement, player comparison, and stronger coaching decisions.